# Mapeamento eleitoral de São João del-Rei – Eleição presidencial 2026


Dados de entrada (gerados pelo `recorte_sjdr.py` a partir dos arquivos oficiais do TSE):
- `dados_sjdr/votacao_presidente_sjdr.csv`: votos para presidente por seção, 1º e 2º turnos
- `dados_sjdr/locais_votacao_sjdr.csv`: locais de votação de cada seção, com latitude e longitude

## 0. Bibliotecas e leitura dos dados

Lemos tudo como texto (`dtype=str`) para não perder nada na leitura. Depois convertemos só as colunas numéricas que vamos usar.

In [1]:
import pandas as pd

votos = pd.read_csv("dados_sjdr/votacao_presidente_sjdr.csv", sep=";", dtype=str)
locais = pd.read_csv("dados_sjdr/locais_votacao_sjdr.csv", sep=";", dtype=str)

votos["QT_VOTOS"] = votos["QT_VOTOS"].astype(int)

print("votos:", votos.shape, "| locais:", locais.shape)

votos: (2706, 26) | locais: (482, 41)


## Funções e definições usadas nos dois turnos

In [2]:
import numpy as np
import folium
import geopandas as gpd

# Colunas do arquivo de locais que levamos para a tabela de votos no cruzamento
colunas_local = ["NR_ZONA", "NR_SECAO", "NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO",
                 "DS_ENDERECO", "NM_BAIRRO", "NR_LATITUDE", "NR_LONGITUDE"]


def juntar_textos(valores):
    """Junta textos diferentes sem repetir (ex.: dois nomes do mesmo prédio)."""
    unicos = [v for v in dict.fromkeys(valores) if isinstance(v, str) and v.strip()]
    return " / ".join(unicos)


def pct_br(valor):
    """57.6 -> '57,6%' (vírgula decimal, padrão brasileiro)."""
    return f"{valor:.1f}%".replace(".", ",")


def num_br(valor):
    """2074 -> '2.074' (ponto como separador de milhar, padrão brasileiro)."""
    return f"{valor:,}".replace(",", ".")


# Opacidade do mapa de fundo (0 = invisível, 1 = normal). Ajuste olhando o resultado.
OPACIDADE_FUNDO = 0.6


# Estilo da "caixa" do tooltip (CSS). Fundo claro para as cores dos partidos terem contraste.
ESTILO_TOOLTIP = """
    background-color: #ffffff;
    color: #222222;
    font-family: 'Helvetica Neue', Arial, sans-serif;
    font-size: 13px;
    border-radius: 8px;
    border: 1px solid #cccccc;
    padding: 10px 12px;
    box-shadow: 3px 3px 10px rgba(0, 0, 0, 0.25);
"""


def mapa_base():
    """Mapa vazio com o fundo usado em todos os mapas (1º e 2º turnos)."""
    # Mapa base: mesmo servidor do notebook da aula (funciona sem chave de API).
    # O CartoDB Positron, recomendado no slide 7 para dados estatísticos, hoje exige chave.
    # Para ter um efeito parecido, deixamos o fundo em escala de cinza (filtro CSS) e mais claro (opacidade):
    # assim a única informação colorida do mapa é o dado eleitoral.
    m = folium.Map(tiles=None)
    folium.TileLayer(
        tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
        attr="&copy; OpenStreetMap contributors",
        name="Mapa base (OpenStreetMap em cinza)",
        opacity=OPACIDADE_FUNDO,
        class_name="fundo-cinza",     # "etiqueta" usada pelo CSS abaixo
    ).add_to(m)
    m.get_root().header.add_child(folium.Element(
        "<style>"
        ".fundo-cinza { filter: grayscale(100%); }"
        ".leaflet-container { background: #ffffff; }"   # fundo branco atrás dos tiles (o padrão é cinza)
        "</style>"
    ))
    return m

## 1º turno – preparação dos dados


O caminho é o mesmo do 2º turno (explicado em detalhe nos Passos 1 a 5 do 2º turno, mais abaixo):
filtrar o turno → pivotar → cruzar com os locais pela chave `NR_ZONA` + `NR_SECAO` → agrupar por coordenada.
A diferença é que no 1º turno há **13 candidatos**, e não 2.

### Ranking em São João del-Rei

Os "principais candidatos" são os **4 mais votados no município** 


In [3]:
votos_1t = votos[votos["NR_TURNO"] == "1"].copy()
locais_1t = locais[locais["NR_TURNO"] == "1"].copy()

BRANCO_NULO = ["VOTO BRANCO", "VOTO NULO"]

# Ranking da cidade
ranking_sjdr = (
    votos_1t[~votos_1t["NM_VOTAVEL"].isin(BRANCO_NULO)]
    .groupby("NM_VOTAVEL")["QT_VOTOS"].sum()
    .sort_values(ascending=False)
    .reset_index()
)
ranking_sjdr["PCT_SJDR"] = 100 * ranking_sjdr["QT_VOTOS"] / ranking_sjdr["QT_VOTOS"].sum()
ranking_sjdr["POS_SJDR"] = range(1, len(ranking_sjdr) + 1)

ranking_sjdr.round(1)

,NM_VOTAVEL,QT_VOTOS,PCT_SJDR,POS_SJDR
0,FLAVIO NANTES BOLSONARO,25618,49.0,1
1,LUIZ INÁCIO LULA DA SILVA,21946,41.9,2
2,AUGUSTO JORGE CURY,1858,3.6,3
3,RENAN ANTONIO FERREIRA DOS SANTOS,1157,2.2,4
4,RONALDO RAMOS CAIADO,1102,2.1,5
5,ROMEU ZEMA NETO,466,0.9,6
6,SAMARA MARTINS DA SILVA FEITOSA,76,0.1,7
7,HERTZ DA CONCEICAO DIAS,48,0.1,8
8,CLARIANA ZACARKIM BARAO,21,0.0,9
9,EDMILSON SILVA COSTA,17,0.0,10


In [4]:
# Os 4 principais = os 4 mais votados na cidade
TOP4 = ranking_sjdr["NM_VOTAVEL"].head(4).tolist()

# Nomes curtos para títulos, legendas e tooltips
NOMES_CURTOS = {
    "FLAVIO NANTES BOLSONARO": "Flávio Bolsonaro",
    "LUIZ INÁCIO LULA DA SILVA": "Lula",
    "AUGUSTO JORGE CURY": "Augusto Cury",
    "RENAN ANTONIO FERREIRA DOS SANTOS": "Renan Santos",
}
print("TOP4:", [NOMES_CURTOS.get(n, n) for n in TOP4])

# Conferência: a distância entre o 4º e o 5º colocado na cidade
quarto, quinto = ranking_sjdr.iloc[3], ranking_sjdr.iloc[4]
print(f"4º: {quarto['NM_VOTAVEL']} ({quarto['QT_VOTOS']} votos) | "
      f"5º: {quinto['NM_VOTAVEL']} ({quinto['QT_VOTOS']} votos) | "
      f"diferença: {quarto['QT_VOTOS'] - quinto['QT_VOTOS']} votos")

TOP4: ['Flávio Bolsonaro', 'Lula', 'Augusto Cury', 'Renan Santos']
4º: RENAN ANTONIO FERREIRA DOS SANTOS (1157 votos) | 5º: RONALDO RAMOS CAIADO (1102 votos) | diferença: 55 votos


### Votos por prédio (pivotar, cruzar e agrupar por coordenada)

Mesmas decisões do 2º turno: só as seções **principais** (as agregadas votam na urna da principal) e
**um ponto por coordenada** (prédios com mais de um código de local viram um ponto só).

In [5]:
# Pivotar: uma linha por seção, uma coluna por opção de voto (13 candidatos + branco + nulo)
por_secao_1t = votos_1t.pivot_table(
    index=["NR_ZONA", "NR_SECAO"], columns="NM_VOTAVEL", values="QT_VOTOS",
    aggfunc="sum", fill_value=0,
).reset_index()
por_secao_1t.columns.name = None
OPCOES_1T = [c for c in por_secao_1t.columns if c not in ("NR_ZONA", "NR_SECAO")]
CANDIDATOS_1T = [c for c in OPCOES_1T if c not in BRANCO_NULO]

# Cruzar com os locais (chaves padronizadas, só seções principais)
for tabela in (por_secao_1t, locais_1t):
    for coluna in ("NR_ZONA", "NR_SECAO"):
        tabela[coluna] = tabela[coluna].str.strip()
locais_principais_1t = locais_1t[locais_1t["DS_TIPO_SECAO_AGREGADA"] == "Principal"]

secoes_1t = por_secao_1t.merge(
    locais_principais_1t[colunas_local], on=["NR_ZONA", "NR_SECAO"],
    how="left", validate="one_to_one",
)
print("seções com voto:", len(secoes_1t), "| seções sem lat/long:", secoes_1t["NR_LATITUDE"].isna().sum())

for coluna in ("NR_LATITUDE", "NR_LONGITUDE"):
    secoes_1t[coluna] = secoes_1t[coluna].str.replace(",", ".").astype(float)

# Agrupar por coordenada, somando todas as opções de voto
por_local_1t = secoes_1t.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(
    NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", juntar_textos),
    NR_LOCAL_VOTACAO=("NR_LOCAL_VOTACAO", juntar_textos),
    NM_BAIRRO=("NM_BAIRRO", juntar_textos),
    QT_SECOES=("NR_SECAO", "count"),
    SECOES=("NR_SECAO", lambda s: ", ".join(sorted(s, key=int))),
    **{opcao: (opcao, "sum") for opcao in OPCOES_1T},
)
print("pontos no mapa:", len(por_local_1t))

seções com voto: 218 | seções sem lat/long: 0
pontos no mapa: 62


### Percentuais dos 4 principais e líder de cada prédio

Percentuais sobre os **votos válidos** do prédio (todos os 13 candidatos, sem brancos e nulos).
O **líder** é o candidato mais votado no prédio entre todos os 13 (não só entre os 4).

In [6]:
por_local_1t["VALIDOS"] = por_local_1t[CANDIDATOS_1T].sum(axis=1)
por_local_1t["TOTAL"] = por_local_1t[OPCOES_1T].sum(axis=1)

for candidato in TOP4:
    por_local_1t[f"PCT_{NOMES_CURTOS[candidato]}"] = 100 * por_local_1t[candidato] / por_local_1t["VALIDOS"]

por_local_1t["LIDER"] = por_local_1t[CANDIDATOS_1T].idxmax(axis=1).map(lambda n: NOMES_CURTOS.get(n, n))
por_local_1t["PCT_LIDER"] = 100 * por_local_1t[CANDIDATOS_1T].max(axis=1) / por_local_1t["VALIDOS"]

# Conferência 1: os totais por candidato têm que bater com o ranking da cidade
conferencia = por_local_1t[CANDIDATOS_1T].sum().sort_values(ascending=False)
print("totais batem com o ranking:",
      conferencia.astype(int).to_dict() == ranking_sjdr.set_index("NM_VOTAVEL")["QT_VOTOS"].to_dict())

# Conferência 2: empate no 1º lugar (idxmax escolheria um dos dois sem avisar)
maximos = por_local_1t[CANDIDATOS_1T].max(axis=1)
empates_1t = (por_local_1t[CANDIDATOS_1T].eq(maximos, axis=0).sum(axis=1) > 1).sum()
print("prédios com empate no 1º lugar:", empates_1t)

print(por_local_1t["LIDER"].value_counts())
colunas_pct = [f"PCT_{NOMES_CURTOS[c]}" for c in TOP4]
por_local_1t[["NM_LOCAL_VOTACAO", "TOTAL", "LIDER"] + colunas_pct].round(1).head()

totais batem com o ranking: True
prédios com empate no 1º lugar: 0
LIDER
Flávio Bolsonaro    43
Lula                19
Name: count, dtype: int64


,NM_LOCAL_VOTACAO,TOTAL,LIDER,PCT_Flávio Bolsonaro,PCT_Lula,PCT_Augusto Cury,PCT_Renan Santos
0,CAPELA DA C0MUNIDADE DO ZUEIRA,84,Flávio Bolsonaro,85.5,2.4,7.2,3.6
1,CAPELA DO VALO NOVO,73,Flávio Bolsonaro,66.7,26.1,1.4,0.0
2,SALÃO DA CAPELA DE SÃO BENEDITO,56,Flávio Bolsonaro,56.5,37.0,2.2,0.0
3,SALÃO COMUNITARIO DO CAQUENDE,91,Lula,30.7,61.4,3.4,0.0
4,SALÃO PAROQUIAL DE CRUZEIRO DA BARRA,58,Flávio Bolsonaro,58.0,32.0,2.0,0.0


## 1º turno – mapas de calor

Um mapa para cada um dos 4 principais candidatos, feito com o **`HeatMap`** (plugin do Folium). Cada local de
votação vira um "borrão" de cor, e borrões vizinhos se somam: o mapa mostra **em que regiões** o candidato foi forte,
sem precisar de fronteiras (não temos os polígonos das seções, só o ponto de cada local de votação).

**O que esquenta o mapa: votos acima do esperado.** Em cada local, o *esperado* é quanto o candidato teria se o local
votasse igual à cidade: `votos válidos do local × % do candidato na cidade`. O peso do local é a diferença entre os
votos que ele teve e esse esperado. Exemplo: um local com 1.000 válidos esperaria 419 votos do Lula (41,9% na cidade);
se ele teve 500, entra com peso +81.

Descartamos duas opções mais simples, que deixariam o mapa enganoso:
- **Votos brutos do candidato:** o mapa mostraria onde há mais eleitores, e não onde o candidato foi melhor. Os votos de
  cada candidato acompanham o tamanho do local (correlação de 0,95 a 0,98, na tabela abaixo), então os quatro mapas
  sairiam quase iguais, todos quentes no centro.
- **% dos votos válidos:** o `HeatMap` *soma* os pesos dos locais próximos. No centro, onde há muitos locais juntos, as
  porcentagens somadas deixariam a região quente para qualquer candidato. E um povoado com 80 votos válidos, onde 3 votos
  mudam muito a %, pesaria tanto quanto uma escola com 3 mil.

**Locais abaixo da média entram com peso negativo.** O plugin junta os locais que caem no mesmo quadrado da tela
(uns 350 m no zoom 13; quadrados maiores com o zoom afastado) e soma os pesos. Com o peso negativo, o que ficou abaixo
da média desconta o que ficou acima, e só esquenta onde o saldo da região é positivo. Testamos deixar só os positivos:
com o zoom afastado, a cidade inteira ficava quente para todos os candidatos.

**Decisões de visualização:**
- **Cores:** degradê sequencial, do claro (pouco acima da média) ao escuro (muito acima), na cor do partido: **azul**
  para Flávio (PL, cujas cores são verde, amarelo, azul e branco; é também o azul dele no mapa do 2º turno),
  **vermelho** para Lula (PT), **laranja** para Augusto Cury (Avante: laranja, ciano e branco) e **amarelo → preto**
  para Renan Santos (Missão: preto, branco e amarelo). Azul, vermelho e laranja vêm das paletas sequenciais do
  ColorBrewer (`Blues`, `Reds` e `Oranges`); a do Missão foi montada à mão, porque o ColorBrewer não tem amarelo → preto.
- **Escala e classes:** o calor não tem classes, é um degradê contínuo de 4 tons. A escala é **relativa**: o tom mais
  escuro é o lugar com o maior saldo *daquele* candidato. Por isso a intensidade não se compara entre os mapas: o maior
  saldo de um local é de +199,6 votos para o Flávio e de +18,5 para o Cury.
- **Raio:** mantivemos o padrão do plugin (raio de 25 px e desfoque de 15 px). No zoom 13, cada borrão tem uns 700 m de
  raio, mais ou menos a escala de um bairro.
- **Opacidade:** o mapa de fundo é o mesmo do 2º turno (OpenStreetMap em cinza, opacidade 0,6). A camada de calor fica
  com opacidade 0,75: com 1, o centro do calor escondia as ruas em volta do local; com 0,5, os tons claros sumiam no
  fundo cinza. O `HeatMap` não tem opção de opacidade, então ela é aplicada por CSS na camada.
- **Enquadramento:** o mapa abre na sede, onde estão 91% dos votos válidos. Com o zoom afastado, a cidade vira uma
  mancha só; os distritos e povoados aparecem ao afastar o zoom.
- **Interatividade:** o calor é uma imagem e não aceita tooltip. Por isso cada local também vira um **ponto branco**
  (`GeoJson` + `CircleMarker`) com um **`GeoJsonTooltip`** em HTML/CSS: nome do local, zona e seções, bairro, candidato
  líder, % dos 4 principais (com o candidato do mapa em destaque) e quantos votos ele teve acima ou abaixo do esperado.
- **Controle de camadas:** `folium.LayerControl` para ligar e desligar o calor e os pontos.
- **Limitação:** como o plugin agrupa os locais em quadrados da tela, o calor muda um pouco com o zoom e o
  enquadramento (dois locais vizinhos podem cair no mesmo quadrado ou não). Os números exatos de cada local estão no
  tooltip.

In [7]:
# % de cada candidato nos votos válidos da cidade (vem do ranking)
PCT_CIDADE_1T = ranking_sjdr.set_index("NM_VOTAVEL")["PCT_SJDR"]

# Votos acima do esperado: quanto o candidato teve a mais (ou a menos, se negativo)
# do que teria se o local votasse igual à cidade
for candidato in TOP4:
    nome = NOMES_CURTOS[candidato]
    esperado = por_local_1t["VALIDOS"] * PCT_CIDADE_1T[candidato] / 100
    por_local_1t[f"ACIMA_{nome}"] = por_local_1t[candidato] - esperado

# Sede × resto do município: fora da sede = locais cujo bairro, no cadastro do TSE,
# é distrito, povoado ou zona rural
PADRAO_FORA_DA_SEDE = "DISTRITO|POVOADO|ZONA RURAL"
FORA_DA_SEDE = por_local_1t["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)

# Conferência: o que um local tem acima da média, outros têm abaixo; somando a cidade toda, dá zero
print("soma dos votos acima do esperado dá zero para os 4:",
      all(abs(por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"].sum()) < 1e-6 for c in TOP4))

# Por que não usar os votos brutos: eles acompanham o tamanho do local (correlação perto de 1)
pd.DataFrame(
    {
        "% na cidade": [PCT_CIDADE_1T[c] for c in TOP4],
        "correlação votos × válidos do local": [por_local_1t[c].corr(por_local_1t["VALIDOS"]) for c in TOP4],
        "locais acima da média": [(por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"] > 0).sum() for c in TOP4],
        "maior saldo de um local (votos)": [por_local_1t[f"ACIMA_{NOMES_CURTOS[c]}"].max() for c in TOP4],
    },
    index=[NOMES_CURTOS[c] for c in TOP4],
).round(2)

soma dos votos acima do esperado dá zero para os 4: True


,% na cidade,correlação votos × válidos do local,locais acima da média,maior saldo de um local (votos)
Flávio Bolsonaro,48.95,0.98,32,199.61
Lula,41.94,0.97,29,193.83
Augusto Cury,3.55,0.95,30,18.54
Renan Santos,2.21,0.95,26,17.63


In [8]:
from folium.plugins import HeatMap

# São João del-Rei tem uma zona eleitoral só (a 328): conferimos em vez de supor
assert votos["NR_ZONA"].nunique() == 1
ZONA = votos["NR_ZONA"].iloc[0]

NOMES_TOP4 = [NOMES_CURTOS[c] for c in TOP4]
PCT_CIDADE_CURTO = {NOMES_CURTOS[c]: PCT_CIDADE_1T[c] for c in TOP4}
PARTIDOS = {"Flávio Bolsonaro": "PL, 22", "Lula": "PT, 13", "Augusto Cury": "Avante, 70", "Renan Santos": "Missão, 14"}

# Degradê de cada candidato na cor do partido, do claro (pouco acima da média) ao escuro (muito acima).
# Azul, vermelho e laranja são das paletas sequenciais do ColorBrewer (Blues, Reds e Oranges).
DEGRADES = {
    "Flávio Bolsonaro": ["#c6dbef", "#6baed6", "#2171b5", "#08306b"],   # PL: azul
    "Lula":             ["#fcbba1", "#fb6a4a", "#cb181d", "#67000d"],   # PT: vermelho
    "Augusto Cury":     ["#fdd0a2", "#fd8d3c", "#d94801", "#7f2704"],   # Avante: laranja
    "Renan Santos":     ["#fff3a0", "#f5c400", "#8a6d00", "#1a1a1a"],   # Missão: amarelo -> preto
}
# Cor do nome no tooltip e na legenda (escura o bastante para ler sobre o branco).
# Lula e Flávio com as mesmas cores do mapa do 2º turno.
COR_NOME = {"Flávio Bolsonaro": "#2166ac", "Lula": "#b2182b", "Augusto Cury": "#d94801", "Renan Santos": "#8a6d00"}

# Parâmetros do HeatMap, em pixels de tela
RAIO_CALOR = 25          # raio de cada "borrão"
DESFOQUE_CALOR = 15      # quanto a borda do borrão se esfuma
OPACIDADE_CALOR = 0.75   # transparência da camada de calor inteira


def cartao_html_1t(local, candidato):
    """HTML do tooltip de um local no mapa de um candidato (nome curto)."""
    lider = local["LIDER"]
    linhas_tabela = "".join(
        f'<tr style="{"font-weight:700; background:#f2f2f2;" if nome == candidato else ""}">'
        f'<td style="color:{COR_NOME[nome]}; padding:1px 4px;">{nome}</td>'
        f'<td style="text-align:right; padding:1px 4px;">{pct_br(local[f"PCT_{nome}"])}</td></tr>'
        for nome in NOMES_TOP4
    )
    acima = int(round(local[f"ACIMA_{candidato}"]))
    if acima > 0:
        saldo = f"<b>{num_br(acima)} votos acima</b> do esperado"
    elif acima < 0:
        saldo = f"{num_br(-acima)} votos abaixo do esperado"
    else:
        saldo = "o esperado para a média da cidade"
    bairro = f'<div style="color:#666; font-size:12px; margin-bottom:8px;">Bairro: {local["NM_BAIRRO"]}</div>' if local["NM_BAIRRO"] else ""
    return f"""
    <div style="min-width:240px; max-width:320px; white-space:normal;">
      <div style="font-weight:700; font-size:14px; margin-bottom:2px;">{local["NM_LOCAL_VOTACAO"]}</div>
      <div style="color:#666; font-size:12px; margin-bottom:2px;">Zona {ZONA} · Seções {local["SECOES"]}</div>
      {bairro}
      <div style="font-weight:700; margin-bottom:4px;">Líder: <span style="color:{COR_NOME.get(lider, '#222222')}">{lider}</span> com {pct_br(local["PCT_LIDER"])}</div>
      <div style="color:#666; font-size:11px;">% dos votos válidos no local:</div>
      <table style="width:100%; font-size:12px; border-collapse:collapse; margin-bottom:6px;">{linhas_tabela}</table>
      <div style="font-size:12px;">{candidato}: {pct_br(local[f"PCT_{candidato}"])} aqui, {pct_br(PCT_CIDADE_CURTO[candidato])} na cidade.<br>Teve {saldo}.</div>
      <div style="color:#666; font-size:12px; margin-top:6px;">{num_br(int(local["TOTAL"]))} votos ({num_br(int(local["TOTAL"] - local["VALIDOS"]))} brancos/nulos)</div>
    </div>"""


def legenda_calor_html(candidato):
    """Legenda: o degradê do calor na cor do partido + o que é cada ponto branco."""
    cores = DEGRADES[candidato]
    return f'''
    <div style="position:fixed; top:12px; right:12px; z-index:9999; width:300px;
                background:rgba(255,255,255,0.92); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,0.3); font-family:Arial, sans-serif; color:#222;">
      <div style="font-weight:700; font-size:13px;">1º turno – <span style="color:{COR_NOME[candidato]};">{candidato}</span> ({PARTIDOS[candidato]})</div>
      <div style="font-size:11px; margin:2px 0 6px;">Onde teve <b>mais votos do que o esperado</b> pela média da cidade
        ({pct_br(PCT_CIDADE_CURTO[candidato])} dos válidos)</div>
      <div style="height:12px; border:1px solid #999; opacity:{OPACIDADE_CALOR};
                  background:linear-gradient(to right, #ffffff, {", ".join(cores)});"></div>
      <div style="display:flex; justify-content:space-between; font-size:11px; margin-top:2px;">
        <span>pouco acima</span><span>muito acima</span>
      </div>
      <div style="display:flex; align-items:center; gap:6px; font-size:11px; margin-top:6px;">
        <div style="width:10px; height:10px; border-radius:50%; background:#ffffff; border:1px solid #555;"></div>
        Local de votação (passe o mouse para ver os detalhes)
      </div>
      <div style="font-size:11px; color:#666; margin-top:4px;">Afaste o zoom para ver os distritos e povoados.</div>
    </div>'''


def mapa_calor_1t(dados, candidato):
    """Mapa do 1º turno de um candidato (nome curto): calor dos votos acima do esperado + pontos com tooltip."""
    m = mapa_base()
    coluna = f"ACIMA_{candidato}"
    cores = DEGRADES[candidato]

    # Camada 1 – calor. O HeatMap junta os locais que caem no mesmo pedaço da tela e SOMA os pesos.
    # Os locais abaixo da média entram com peso negativo: quando o zoom junta vários locais,
    # o que ficou abaixo desconta o que ficou acima, e só esquenta onde o saldo é positivo.
    # min_opacity quase zero: sem isso, até um saldo negativo apareceria como uma mancha clara.
    HeatMap(
        dados[["NR_LATITUDE", "NR_LONGITUDE", coluna]].values.tolist(),
        name="Calor: votos acima do esperado",
        radius=RAIO_CALOR,
        blur=DESFOQUE_CALOR,
        min_opacity=0.001,
        gradient={0.25: cores[0], 0.5: cores[1], 0.75: cores[2], 1.0: cores[3]},
    ).add_to(m)
    # O HeatMap não tem opção de opacidade: aplicamos por CSS na camada (classe criada pelo plugin)
    m.get_root().header.add_child(folium.Element(
        f"<style>.leaflet-heatmap-layer {{ opacity: {OPACIDADE_CALOR}; }}</style>"
    ))

    # Camada 2 – um ponto por local, com o tooltip (o HeatMap é uma imagem e não aceita tooltip)
    pontos = dados.copy()
    pontos["CARTAO"] = pontos.apply(cartao_html_1t, axis=1, candidato=candidato)
    gdf = gpd.GeoDataFrame(
        pontos[["CARTAO"]],
        geometry=gpd.points_from_xy(pontos["NR_LONGITUDE"], pontos["NR_LATITUDE"]),
        crs="EPSG:4326",
    )
    folium.GeoJson(
        gdf,
        name="Locais de votação",
        marker=folium.CircleMarker(),
        style_function=lambda f: {
            "radius": 5,
            "fill": True,
            "fillColor": "#ffffff",
            "fillOpacity": 0.9,
            "color": "#555555",
            "weight": 1,
        },
        highlight_function=lambda f: {"weight": 3, "color": "#000000"},
        tooltip=folium.GeoJsonTooltip(fields=["CARTAO"], labels=False, sticky=True, style=ESTILO_TOOLTIP),
    ).add_to(m)

    folium.LayerControl(position="bottomleft", collapsed=False).add_to(m)   # liga/desliga cada camada
    m.get_root().html.add_child(folium.Element(legenda_calor_html(candidato)))
    # Abre enquadrando a sede (com o zoom afastado, a cidade vira uma mancha só); distritos e povoados
    # aparecem ao afastar o zoom. max_zoom=13: numa tela grande, não aproxima além do nível dos bairros.
    na_sede = ~dados["NM_BAIRRO"].str.contains(PADRAO_FORA_DA_SEDE)
    oeste, sul, leste, norte = gdf[na_sede.values].total_bounds
    m.fit_bounds([[sul, oeste], [norte, leste]], max_zoom=13)
    return m

In [9]:
from pathlib import Path
Path("mapas").mkdir(exist_ok=True)

# Um mapa por candidato, salvo em mapas/mapa_1turno_<nome>.html
ARQUIVOS_1T = {"Flávio Bolsonaro": "flavio", "Lula": "lula", "Augusto Cury": "cury", "Renan Santos": "renan"}
mapas_1t = {}
for nome in NOMES_TOP4:
    mapas_1t[nome] = mapa_calor_1t(por_local_1t, nome)
    arquivo = f"mapas/mapa_1turno_{ARQUIVOS_1T[nome]}.html"
    mapas_1t[nome].save(arquivo)
    print("salvo:", arquivo)

salvo: mapas/mapa_1turno_flavio.html
salvo: mapas/mapa_1turno_lula.html
salvo: mapas/mapa_1turno_cury.html


salvo: mapas/mapa_1turno_renan.html


In [10]:
# Números usados nos relatórios dos mapas do 1º turno
sede, fora = por_local_1t[~FORA_DA_SEDE], por_local_1t[FORA_DA_SEDE]
print(f"Locais na sede: {len(sede)}, com {pct_br(100 * sede['VALIDOS'].sum() / por_local_1t['VALIDOS'].sum())} "
      f"dos válidos | em distritos e povoados: {len(fora)}")

# 1) % na sede e fora dela
display(pd.DataFrame(
    {
        "cidade": [PCT_CIDADE_1T[c] for c in TOP4],
        "sede": [100 * sede[c].sum() / sede["VALIDOS"].sum() for c in TOP4],
        "distritos e povoados": [100 * fora[c].sum() / fora["VALIDOS"].sum() for c in TOP4],
    },
    index=NOMES_TOP4,
).round(1))

# 2) Como a % de cada candidato varia entre os locais
display(por_local_1t[[f"PCT_{n}" for n in NOMES_TOP4]].describe().loc[["min", "25%", "50%", "75%", "max"]].round(1))

# 3) Saldo de votos acima do esperado por bairro (o que ficou abaixo desconta), nos bairros com mais de 1.000 válidos
por_bairro = por_local_1t.groupby("NM_BAIRRO").agg(
    VALIDOS=("VALIDOS", "sum"), **{nome: (f"ACIMA_{nome}", "sum") for nome in NOMES_TOP4}
)
display(por_bairro[por_bairro["VALIDOS"] > 1000].sort_values("Lula", ascending=False).round(0))

# 4) Os locais mais acima do esperado de cada candidato e os extremos de %
for nome in NOMES_TOP4:
    print(f"\n=== {nome} ===")
    print(por_local_1t.nlargest(4, f"ACIMA_{nome}")[["NM_LOCAL_VOTACAO", "NM_BAIRRO", "VALIDOS", f"PCT_{nome}", f"ACIMA_{nome}"]]
          .round(1).to_string(index=False))
    for rotulo, i in (("menor %", por_local_1t[f"PCT_{nome}"].idxmin()), ("maior %", por_local_1t[f"PCT_{nome}"].idxmax())):
        r = por_local_1t.loc[i]
        print(f"{rotulo}: {r['NM_LOCAL_VOTACAO']} ({r['NM_BAIRRO']}, {r['VALIDOS']} válidos): {pct_br(r[f'PCT_{nome}'])}")

Locais na sede: 43, com 91,4% dos válidos | em distritos e povoados: 19


,cidade,sede,distritos e povoados
Flávio Bolsonaro,49.0,49.2,46.4
Lula,41.9,41.7,44.6
Augusto Cury,3.6,3.5,3.7
Renan Santos,2.2,2.3,1.2


,PCT_Flávio Bolsonaro,PCT_Lula,PCT_Augusto Cury,PCT_Renan Santos
min,20.8,2.4,1.2,0.0
25%,42.1,36.6,2.6,1.2
50%,49.7,41.3,3.5,2.0
75%,54.8,47.9,4.2,2.5
max,85.5,61.7,7.2,5.9


,VALIDOS,Flávio Bolsonaro,Lula,Augusto Cury,Renan Santos
NM_BAIRRO,,,,,
TEJUCO,5971,-353.0,356.0,22.0,-11.0
SENHOR DOS MONTES,2287,-224.0,196.0,12.0,4.0
DOM BOSCO / FABRICAS,1765,-188.0,194.0,-11.0,10.0
DISTRITO DO RIO DAS MORTES,1239,-180.0,183.0,5.0,-14.0
BELA VISTA,1214,-92.0,92.0,3.0,6.0
BONFIM,2574,-123.0,73.0,12.0,20.0
GUARDA-MOR,1160,-34.0,47.0,2.0,-4.0
VILA SANTA TEREZINHA,1207,21.0,-33.0,-2.0,0.0
FABRICAS,4419,63.0,-59.0,-16.0,1.0



=== Flávio Bolsonaro ===
                                 NM_LOCAL_VOTACAO      NM_BAIRRO  VALIDOS  PCT_Flávio Bolsonaro  ACIMA_Flávio Bolsonaro
         ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS     MATOSINHOS     2207                  58.0                   199.6
                  COLEGIO NOSSA SENHORA DAS DORES         CENTRO     3151                  54.1                   163.5
SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI     MATOSINHOS     1998                  55.7                   134.9
                          COLÉGIO TIRADENTES PMMG VILA BELIZÁRIO     1275                  57.0                   102.9
menor %: SALÃO COMUNITARIO DO CANELA (POVOADO DE CANELA, 120 válidos): 20,8%
maior %: CAPELA DA C0MUNIDADE DO ZUEIRA (POVOADO DO ZUEIRA, 83 válidos): 85,5%

=== Lula ===
                                              NM_LOCAL_VOTACAO                  NM_BAIRRO  VALIDOS  PCT_Lula  ACIMA_Lula
UFSJ - CAMPUS DOM BOSCO / SEÇÃO ESPECIAL-UFSJ CAMPUS DOM BOSCO       DOM BOSCO / FA

In [11]:
mapas_1t["Flávio Bolsonaro"]

#### Relatório – Mapa do 1º turno: Flávio Bolsonaro

**O fato**
- **Venceu na cidade com 49,0% dos válidos** e liderou em 43 dos 62 locais.
- **O calor fica no Centro e a leste dele.** Matosinhos é, de longe, o bairro com maior saldo (+524 votos acima do
  esperado), seguido de Centro (+140), Vila Belizário (+103) e Pio XII (+98). Os locais mais quentes são a E.E. Governador
  Milton Campos (58,0% dos válidos, +200 votos) e o SENAI (55,7%, +135), os dois em Matosinhos, o Colégio Nossa Senhora
  das Dores (54,1%, +164), no Centro, e o Colégio Tiradentes PMMG (57,0%, +103), na Vila Belizário.
- **A oeste, o mapa fica sem cor:** no Tejuco (−353), em Senhor dos Montes (−224) e no Bonfim (−123) ele ficou abaixo da
  média. É o mesmo leste × oeste que aparece no mapa do 2º turno.
- **Sede × distritos e povoados:** 49,2% na sede e 46,4% fora dela. Os extremos estão em povoados pequenos, onde poucos
  votos mudam muito a %: Zueira (85,5%, com 83 válidos) e Canela (20,8%, com 120).

**O design**
- **Paleta:** degradê azul do ColorBrewer (`Blues`). Azul é uma das cores do PL e é a cor do Flávio no mapa do 2º turno.
- **Classes e escala:** degradê contínuo, sem classes; escala relativa, com o azul mais escuro no maior saldo do mapa (os
  locais de Matosinhos, que se somam).
- **Folium:** `HeatMap` (calor), `GeoJson` + `CircleMarker` com `GeoJsonTooltip` (pontos), `LayerControl` e legenda em
  HTML. Raio, opacidade e enquadramento como nas decisões acima.

In [12]:
mapas_1t["Lula"]

#### Relatório – Mapa do 1º turno: Lula

**O fato**
- **Segundo lugar na cidade, com 41,9% dos válidos;** liderou em 19 dos 62 locais.
- **O mapa é o espelho do mapa do Flávio.** O calor fica a oeste do Centro, no Tejuco (+356 votos acima do esperado) e
  em Senhor dos Montes (+196), e ao norte, na Bela Vista (+92) e no Campus Dom Bosco da UFSJ, o local com o maior saldo
  (52,9% dos válidos, +194). Matosinhos (−515) e Centro (−82), onde o Flávio esquenta, ficam sem cor.
- **Fora da sede, o destaque é o Distrito do Rio das Mortes:** a E.E. Evandro Ávila deu 56,7% ao Lula (+183 votos). Nos
  distritos e povoados como um todo ele teve 44,6%, contra 41,7% na sede.
- **Extremos:** Canela (61,7%) e Zueira (2,4%), dois povoados pequenos.

**O design**
- **Paleta:** degradê vermelho do ColorBrewer (`Reds`): a cor do PT e a mesma do Lula no mapa do 2º turno.
- **Classes e escala:** degradê contínuo, sem classes; escala relativa, com o vermelho mais escuro no maior saldo do mapa
  (as duas escolas do Tejuco, que se somam, e o Campus Dom Bosco).
- **Folium:** `HeatMap` (calor), `GeoJson` + `CircleMarker` com `GeoJsonTooltip` (pontos), `LayerControl` e legenda em
  HTML. Raio, opacidade e enquadramento como nas decisões acima.

In [13]:
mapas_1t["Augusto Cury"]

#### Relatório – Mapa do 1º turno: Augusto Cury

**O fato**
- **Terceiro lugar, com 3,6% dos válidos, e um voto bem espalhado:** em metade dos locais ele teve entre 2,6% e 4,2%, e
  o maior saldo de um local é de só +18,5 votos.
- **O calor aparece dos dois lados da cidade,** sem o padrão leste × oeste dos dois primeiros: Matosinhos (+31), Tejuco
  (+22), Pio XII (+19), Senhor dos Montes e Bonfim (+12 cada). Os locais mais quentes são escolas grandes, onde uma dúzia
  de votos a mais já se destaca: E.M. Pio XII (4,5%), Obras Sociais Fé e Alegria (5,3%, no Tejuco) e E.E. Iago Pimentel
  (4,2%, no Tejuco).
- **O ponto fraco é o Centro** (−52 votos). Sede e distritos têm resultado igual (3,5% × 3,7%).
- **Cuidado na leitura:** com tão poucos votos por local, parte dessas diferenças pode ser acaso. Os extremos (1,2% no
  Engenho de Serra e 7,2% no Zueira) estão em povoados com uns 80 votos válidos.

**O design**
- **Paleta:** degradê laranja do ColorBrewer (`Oranges`): laranja é a cor principal do Avante.
- **Classes e escala:** degradê contínuo, sem classes; escala relativa. Atenção: aqui o laranja mais escuro equivale a
  poucas dezenas de votos, enquanto nos mapas de Flávio e Lula o tom mais escuro equivale a centenas.
- **Folium:** `HeatMap` (calor), `GeoJson` + `CircleMarker` com `GeoJsonTooltip` (pontos), `LayerControl` e legenda em
  HTML. Raio, opacidade e enquadramento como nas decisões acima.

In [14]:
mapas_1t["Renan Santos"]

#### Relatório – Mapa do 1º turno: Renan Santos

**O fato**
- **Quarto lugar, com 2,2% dos válidos**, só 55 votos à frente de Ronaldo Caiado (2,1%).
- **Voto mais urbano:** 2,3% na sede e 1,2% nos distritos e povoados, quase a metade.
- **Os pontos mais quentes são o Bonfim e o Tejuco:** o Centro Social São Francisco (Bonfim) deu 5,8% a ele (+17 votos) e
  a E.E. Iago Pimentel (Tejuco), 2,9% (+18). Também aparecem a E.M. Menino Jesus (Solar da Serra, 3,5%) e a E.M. Domingos
  Horta (Rio Acima, 5,9%, a maior % dele na cidade). Em Matosinhos ele ficou abaixo da média (−27).
- **Mesma ressalva do Cury:** são diferenças de 10 a 20 votos por local, e parte delas pode ser acaso.

**O design**
- **Paleta:** degradê amarelo → preto, as cores do Missão, montado à mão (o ColorBrewer não tem essa sequência). O claro
  continua amarelo e o escuro vai para o preto, então a ordem "pouco → muito" se lê pela claridade.
- **Classes e escala:** degradê contínuo, sem classes; escala relativa, como no mapa do Cury: o preto equivale a poucas
  dezenas de votos.
- **Folium:** `HeatMap` (calor), `GeoJson` + `CircleMarker` com `GeoJsonTooltip` (pontos), `LayerControl` e legenda em
  HTML. Raio, opacidade e enquadramento como nas decisões acima.

## 2º turno

### Passo 1 – Filtrar o 2º turno

Os dois arquivos trazem os dois turnos. Ficamos só com as linhas do 2º.

In [15]:
votos_2t = votos[votos["NR_TURNO"] == "2"].copy()
locais_2t = locais[locais["NR_TURNO"] == "2"].copy()

print("linhas de votos:", len(votos_2t), "| linhas de locais:", len(locais_2t))
votos_2t.groupby("NM_VOTAVEL")["QT_VOTOS"].sum()

linhas de votos: 855 | linhas de locais: 241


NM_VOTAVEL
FLAVIO NANTES BOLSONARO      27777
LUIZ INÁCIO LULA DA SILVA    27644
VOTO BRANCO                    784
VOTO NULO                     1278
Name: QT_VOTOS, dtype: int64

### Passo 2 – Pivotar: uma linha por seção

Hoje cada linha é "seção × candidato" (4 linhas por seção: Flávio, Lula, branco e nulo).
O `pivot_table` transforma isso em **uma linha por seção**, com uma coluna para cada opção de voto.

In [16]:
por_secao = votos_2t.pivot_table(
    index=["NR_ZONA", "NR_SECAO"],   # o que vira linha
    columns="NM_VOTAVEL",            # o que vira coluna
    values="QT_VOTOS",               # o que vai dentro das células
    aggfunc="sum",
    fill_value=0,                    # se um candidato teve 0 votos na seção, a linha não existe no TSE
).reset_index()
por_secao.columns.name = None

# Nomes curtos para as colunas
por_secao = por_secao.rename(columns={
    "FLAVIO NANTES BOLSONARO": "FLAVIO",
    "LUIZ INÁCIO LULA DA SILVA": "LULA",
    "VOTO BRANCO": "BRANCO",
    "VOTO NULO": "NULO",
})

print("seções com votos:", len(por_secao))
por_secao.head()

seções com votos: 218


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO
0,328,100,108,171,6,6
1,328,102,93,151,4,6
2,328,103,118,138,5,4
3,328,104,116,144,4,6
4,328,105,119,153,3,12


### Passo 3 – Cruzar votos com locais (chave: `NR_ZONA` + `NR_SECAO`)

A tabela de votos não tem latitude/longitude e a de locais não tem votos. Juntamos as duas pela chave **zona + seção**.

**Padronização das chaves:** as duas colunas precisam estar no mesmo formato nas duas tabelas
(texto sem espaços). Se uma tivesse `"69"` e a outra `69` ou `"069"`, o `merge` não encontraria o par.

**Seções agregadas:** no arquivo de locais há seções do tipo *Agregada*. Os eleitores delas votam na urna
de outra seção (a *Principal*), então elas não têm votos próprios: os votos já estão somados na principal.
Usamos só as principais para não contar nada duas vezes.

In [17]:
# Padronizando as chaves: texto, sem espaços extras
for tabela in (por_secao, locais_2t):
    for coluna in ("NR_ZONA", "NR_SECAO"):
        tabela[coluna] = tabela[coluna].str.strip()

print(locais_2t["DS_TIPO_SECAO_AGREGADA"].value_counts())

locais_principais = locais_2t[locais_2t["DS_TIPO_SECAO_AGREGADA"] == "Principal"]

secoes = por_secao.merge(
    locais_principais[colunas_local],
    on=["NR_ZONA", "NR_SECAO"],
    how="left",                 # mantém todas as seções com voto
    validate="one_to_one",      # dá erro se alguma seção aparecer repetida
)

# Conferência: nenhuma seção com voto pode ficar sem localização
print("seções sem lat/long:", secoes["NR_LATITUDE"].isna().sum())
secoes.head()

DS_TIPO_SECAO_AGREGADA
Principal    218
Agregada      23
Name: count, dtype: int64
seções sem lat/long: 0


,NR_ZONA,NR_SECAO,FLAVIO,LULA,BRANCO,NULO,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,DS_ENDERECO,NM_BAIRRO,NR_LATITUDE,NR_LONGITUDE
0,328,100,108,171,6,6,1104,ESCOLA MUNICIPAL DR. KLEBER VASQUES FILGUEIRAS,"RUA RUTH NEDER ISSA, N 50",TEJUCO,"-21,1416078","-44,276157"
1,328,102,93,151,4,6,1171,CENTRO SOCIAL SAO FRANCISCO,"RUA ESCRIVAO FARNESE SILVA, S/N",BONFIM,"-21,1435736","-44,2583195"
2,328,103,118,138,5,4,1171,CENTRO SOCIAL SAO FRANCISCO,"RUA ESCRIVAO FARNESE SILVA, S/N",BONFIM,"-21,1435736","-44,2583195"
3,328,104,116,144,4,6,1180,ESCOLA ESTADUAL INACIO PASSOS,"PRACA DR GUILHERME MILWARD, S/N",BONFIM,"-21,140479","-44,25804"
4,328,105,119,153,3,12,1180,ESCOLA ESTADUAL INACIO PASSOS,"PRACA DR GUILHERME MILWARD, S/N",BONFIM,"-21,140479","-44,25804"


O TSE escreve as coordenadas com vírgula decimal (`-21,1374636`). Convertemos para número com ponto.

In [18]:
for coluna in ("NR_LATITUDE", "NR_LONGITUDE"):
    secoes[coluna] = secoes[coluna].str.replace(",", ".").astype(float)

secoes[["NR_LATITUDE", "NR_LONGITUDE"]].describe()

,NR_LATITUDE,NR_LONGITUDE
count,218.000000,218.000000
mean,-21.144832,-44.263048
std,0.048779,0.051266
min,-21.374600,-44.542755
25%,-21.140593,-44.265646
50%,-21.134055,-44.254490
75%,-21.126847,-44.240083
max,-21.084874,-44.072441


### Passo 4 – Agrupar por ponto no mapa (coordenada)

Várias seções funcionam no mesmo prédio e têm **a mesma coordenada**. Se cada seção virasse um círculo,
eles ficariam empilhados no mapa.

Primeiro tentamos agrupar pelo **código do local** (`NR_LOCAL_VOTACAO`), mas o mapa ainda mostrou círculos
um dentro do outro: alguns prédios aparecem no TSE com **dois códigos de local diferentes e a mesma coordenada**
(ex.: UFSJ – Campus Dom Bosco e a "Seção Especial" do mesmo campus; E.E. Tomé Portes del Rei; E.M. Maria Teresa;
E.M. Bom Pastor). Por isso agrupamos pela **coordenada**: um círculo por ponto do mapa, somando todos os votos
daquele lugar. Os nomes e as seções de cada ponto ficam guardados para o tooltip.

In [19]:
por_local = secoes.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(
    NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", juntar_textos),
    NR_LOCAL_VOTACAO=("NR_LOCAL_VOTACAO", juntar_textos),
    NM_BAIRRO=("NM_BAIRRO", juntar_textos),
    DS_ENDERECO=("DS_ENDERECO", "first"),
    FLAVIO=("FLAVIO", "sum"),
    LULA=("LULA", "sum"),
    BRANCO=("BRANCO", "sum"),
    NULO=("NULO", "sum"),
    QT_SECOES=("NR_SECAO", "count"),
    SECOES=("NR_SECAO", lambda s: ", ".join(sorted(s, key=int))),
)

print("pontos no mapa:", len(por_local))
print("códigos de local diferentes:", secoes["NR_LOCAL_VOTACAO"].nunique())

# Os pontos que juntaram mais de um código de local (vale citar no relatório)
por_local[por_local["NR_LOCAL_VOTACAO"].str.contains("/")][["NM_LOCAL_VOTACAO", "NR_LOCAL_VOTACAO", "SECOES"]]

pontos no mapa: 62
códigos de local diferentes: 66


,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,SECOES
30,ESCOLA MUNICIPAL MARIA TERESA,1058 / 1066,"40, 41, 67, 68, 69"
32,ESCOLA MUNICIPAL BOM PASTOR (ANTIGA ESCOLA PIP...,1813 / 1830,"307, 310, 311, 312, 313"
40,ESCOLA ESTADUAL TOMÉ PORTES DEL REI,1775 / 1821,"284, 285, 286, 287, 288, 289, 290, 291, 308, 309"
50,UFSJ - CAMPUS DOM BOSCO / SEÇÃO ESPECIAL-UFSJ ...,1252 / 1686,"125, 126, 127, 133, 134, 149, 345"


### Passo 5 – Percentuais sobre os votos válidos

Votos válidos = votos nos candidatos, sem brancos e nulos. É o critério oficial do TSE para definir o vencedor,
e com ele `% Flávio + % Lula = 100%`, então **50% é exatamente o empate**: o ponto neutro da paleta divergente.

In [20]:
por_local["VALIDOS"] = por_local["FLAVIO"] + por_local["LULA"]
por_local["TOTAL"] = por_local["VALIDOS"] + por_local["BRANCO"] + por_local["NULO"]
por_local["PCT_FLAVIO"] = 100 * por_local["FLAVIO"] / por_local["VALIDOS"]
por_local["PCT_LULA"] = 100 * por_local["LULA"] / por_local["VALIDOS"]

# Conferência: os totais da cidade precisam bater com o Passo 1
print(por_local[["FLAVIO", "LULA", "BRANCO", "NULO"]].sum())

por_local.sort_values("PCT_LULA").head(10)

FLAVIO    27777
LULA      27644
BRANCO      784
NULO       1278
dtype: int64


,NR_LATITUDE,NR_LONGITUDE,NM_LOCAL_VOTACAO,NR_LOCAL_VOTACAO,NM_BAIRRO,DS_ENDERECO,FLAVIO,LULA,BRANCO,NULO,QT_SECOES,SECOES,VALIDOS,TOTAL,PCT_FLAVIO,PCT_LULA
0,-21.374600,-44.413545,CAPELA DA C0MUNIDADE DO ZUEIRA,1589,POVOADO DO ZUEIRA,ZONA RURAL,75,8,1,0,1,186,83,84,90.361446,9.638554
5,-21.331336,-44.493679,SALÃO COMUNITÁRIO DO ENGENHO DE SERRA,1481,POVOADO ENGENHO DE SERRA,ZONA RURAL,63,21,2,2,1,172,84,88,75.000000,25.000000
9,-21.258726,-44.201051,CASA PAROQUIAL CAPELA MORRO GRANDE,1724,ZONA RURAL,POVOADO DO MORRO GRANDE,61,27,4,2,1,225,88,94,69.318182,30.681818
11,-21.237010,-44.374351,SALÃO DA IGREJA DE SÃO JANUÁRIO,1473,POVOADO DE JANUÁRIO,"RUA LUIZ SAVIO DE MOURA, 100 - ZONA RURAL",114,69,4,7,1,171,183,194,62.295082,37.704918
4,-21.331740,-44.443482,SALÃO PAROQUIAL DE CRUZEIRO DA BARRA,1538,POVOADO DE CRUZEIRO DA BARRA,ZONA RURAL,30,20,1,3,1,180,50,54,60.000000,40.000000
61,-21.084874,-44.233308,GRUPO ESCOLAR MUNICIPAL PARADA DO GIAROLA,1759,COLÔNIA DO GIAROLA,"R. AMERICO DEODORO BRIGHENTI, S/N",261,177,3,7,2,"230, 231",438,448,59.589041,40.410959
41,-21.128380,-44.234120,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,1864,MATOSINHOS,"AV. SETE DE SETEMBRO, S/N",1373,951,35,57,8,"319, 320, 321, 322, 323, 324, 325, 326",2324,2416,59.079174,40.920826
30,-21.137464,-44.258335,ESCOLA MUNICIPAL MARIA TERESA,1058 / 1066,CENTRO,"PRACA DOS EXPEDICIONARIOS, S/N",678,476,12,25,5,"40, 41, 67, 68, 69",1154,1191,58.752166,41.247834
46,-21.126847,-44.240350,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,1716,MATOSINHOS,"PRAÇA BOM JESUS DE MATOZINHOS, S/N",1233,871,27,43,8,"217, 218, 219, 220, 221, 222, 223, 224",2104,2174,58.602662,41.397338
49,-21.125265,-44.242422,COLEGIO ATUS (ANTIGA ESCOLA DEZ),1791,JARDIM PAULO CAMPOS,RUA JOSE DE RESENDE CAMPOS 479,767,548,16,30,5,"297, 298, 299, 300, 301",1315,1361,58.326996,41.673004


### Passo 6 – Mapa do 2º turno

**Decisões já tomadas:**
- Um círculo por ponto do mapa, com **área proporcional ao total de votos**.
- **Paleta divergente vermelho ↔ azul** (ColorBrewer `RdBu`): vermelho = Lula (13, PT), azul = Flávio (22, PL).
- A cor representa o **% de Lula sobre os votos válidos**. O branco/neutro fica exatamente em **50% (empate)**.

**Por que não usar a escala automática do Folium:** o padrão divide o intervalo entre o menor (9,6%) e o maior (85,2%)
valor dos dados, o que colocaria o meio da escala em 47,4%. Locais onde Flávio venceu por pouco
ficariam com a cor neutra ou até vermelha. Por isso fixamos os limites de forma simétrica em torno de 50%.

**Interatividade:** os pontos viram um **GeoDataFrame** (geopandas) e são desenhados com
`folium.GeoJson` usando `folium.CircleMarker` como marcador. Isso permite usar o **`GeoJsonTooltip`**.
O tooltip é um **cartão em HTML/CSS** montado no pandas, com: nome do local, zona e seções, bairro,
candidato líder (na cor do partido), uma barra com a proporção Lula × Flávio e o total de votos.

**Classes:** testamos duas opções simétricas em torno de 50%:
- **Opção A:** 6 classes (mesmo número do padrão do Folium), com as pontas abertas: até 30 | 30–40 | 40–50 ‖ 50–60 | 60–70 | 70 ou mais
- **Opção B:** 8 classes de 10 pontos, de 10% a 90%

**Escolhemos a A.** Na B, a classe 80–90% ficou vazia (uma cor na legenda que não aparece no mapa) e as classes
extras só separavam 2 povoados pequenos na ponta azul, diferença quase imperceptível no mapa. Os dados estão
concentrados no meio: 86% dos votos estão em locais entre 40% e 60%. A tabela abaixo mostra quantos locais caem
em cada classe nas duas opções.

**Legenda:** a legenda padrão do Folium (branca) desenha cada faixa com largura proporcional ao valor, então a
faixa "até 30" ocupava um terço da barra e as faixas do meio ficavam espremidas. Fizemos uma legenda em HTML
com as 6 faixas do mesmo tamanho.

**Empates:** dois locais tiveram exatamente o mesmo número de votos para os dois candidatos (Centro Educacional
Frei Seráfico e Unidade Regional de Saúde). Com 50%, eles cairiam na classe 50–60 e pareceriam vitória do Lula.
Por isso recebem uma cor própria, **branco** (o centro neutro da paleta divergente), com um item "Empate" na legenda.
Testamos também cinza, mas ele se confundia com o mapa de fundo, que é cinza; a borda cinza dos círculos
mantém os brancos visíveis.

In [21]:
import branca.colormap as cm

# Cores do ColorBrewer RdBu, do azul (Lula baixo = Flávio alto) para o vermelho (Lula alto)
RDBU_6 = ["#2166ac", "#67a9cf", "#d1e5f0", "#fddbc7", "#ef8a62", "#b2182b"]
RDBU_8 = ["#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#fddbc7", "#f4a582", "#d6604d", "#b2182b"]
VERMELHO_LULA, AZUL_FLAVIO = "#b2182b", "#2166ac"   # as pontas da paleta, usadas também no tooltip

# Bordas das classes (% de Lula nos votos válidos). Repare que 50 é sempre uma borda:
# nenhuma classe mistura local onde Lula venceu com local onde Flávio venceu.
BORDAS_A = [0, 30, 40, 50, 60, 70, 100]          # 6 classes, pontas abertas
BORDAS_B = [10, 20, 30, 40, 50, 60, 70, 80, 90]  # 8 classes de 10 pontos


def raio(total, total_max, raio_max=20):
    """Raio em pixels com ÁREA proporcional aos votos (por isso a raiz quadrada).
    Se o raio fosse proporcional aos votos, um local com o dobro de votos teria 4x a área
    e pareceria muito maior do que é."""
    return 3 + (raio_max - 3) * np.sqrt(total / total_max)


def cartao_html(local):
    """Monta o HTML do tooltip de um ponto do mapa."""
    if local["LULA"] > local["FLAVIO"]:
        lider = f'<span style="color:{VERMELHO_LULA}">Lula</span> lidera com {pct_br(local["PCT_LULA"])}'
    elif local["FLAVIO"] > local["LULA"]:
        lider = f'<span style="color:{AZUL_FLAVIO}">Flávio</span> lidera com {pct_br(local["PCT_FLAVIO"])}'
    else:
        lider = "Empate"

    bairro = f'<div style="color:#666; font-size:12px; margin-bottom:8px;">Bairro: {local["NM_BAIRRO"]}</div>' if local["NM_BAIRRO"] else ""
    return f"""
    <div style="min-width:240px; max-width:320px; white-space:normal;">
      <div style="font-weight:700; font-size:14px; margin-bottom:2px;">{local["NM_LOCAL_VOTACAO"]}</div>
      <div style="color:#666; font-size:12px; margin-bottom:2px;">Zona 328 · Seções {local["SECOES"]}</div>
      {bairro}
      <div style="font-weight:700; margin-bottom:4px;">{lider}</div>
      <div style="display:flex; height:10px; border-radius:5px; overflow:hidden; margin-bottom:4px;">
        <div style="width:{local["PCT_LULA"]:.1f}%; background:{VERMELHO_LULA};"></div>
        <div style="width:{local["PCT_FLAVIO"]:.1f}%; background:{AZUL_FLAVIO};"></div>
      </div>
      <div style="display:flex; justify-content:space-between; font-size:12px;">
        <span style="color:{VERMELHO_LULA}">Lula {pct_br(local["PCT_LULA"])}</span>
        <span style="color:{AZUL_FLAVIO}">Flávio {pct_br(local["PCT_FLAVIO"])}</span>
      </div>
      <div style="color:#666; font-size:12px; margin-top:6px;">{num_br(local["TOTAL"])} votos ({num_br(local["BRANCO"] + local["NULO"])} brancos/nulos)</div>
    </div>"""


def rotulos_classes(bordas):
    """[0, 30, 40, ..., 100] -> ['até 30', '30–40', ..., '70 ou mais']"""
    rotulos = []
    for i, (inicio, fim) in enumerate(zip(bordas[:-1], bordas[1:])):
        if i == 0 and inicio == 0:
            rotulos.append(f"até {fim}")
        elif i == len(bordas) - 2 and fim == 100:
            rotulos.append(f"{inicio} ou mais")
        else:
            rotulos.append(f"{inicio}–{fim}")
    return rotulos


def legenda_html(bordas, cores, titulo, cor_empate=None):
    """Legenda com todas as classes do mesmo tamanho (a legenda padrão do branca é proporcional ao valor)."""
    caixas = "".join(
        f'''<div style="flex:1; text-align:center;">
             <div style="height:14px; background:{cor}; border:1px solid #999;"></div>
             <div style="font-size:11px; margin-top:2px;">{rotulo}</div>
           </div>'''
        for cor, rotulo in zip(cores, rotulos_classes(bordas))
    )
    caixa_empate = ""
    if cor_empate:
        caixa_empate = f'''
      <div style="display:flex; align-items:center; gap:6px; font-size:11px; margin-top:6px;">
        <div style="width:14px; height:14px; background:{cor_empate}; border:1px solid #999;"></div>
        Empate exato (mesmo número de votos)
      </div>'''
    return f'''
    <div style="position:fixed; top:12px; right:12px; z-index:9999; width:340px;
                background:rgba(255,255,255,0.92); padding:8px 10px; border-radius:6px;
                box-shadow:0 1px 6px rgba(0,0,0,0.3); font-family:Arial, sans-serif; color:#222;">
      <div style="font-weight:700; font-size:12px; margin-bottom:6px;">{titulo}</div>
      <div style="display:flex; gap:2px;">{caixas}</div>{caixa_empate}
      <div style="display:flex; justify-content:space-between; font-size:11px; margin-top:4px;">
        <span style="color:{AZUL_FLAVIO};">&#9664; Flávio venceu</span>
        <span style="color:{VERMELHO_LULA};">Lula venceu &#9654;</span>
      </div>
    </div>'''


def mapa_2turno(dados, bordas, cores, titulo, cor_empate="#ffffff"):
    escala = cm.StepColormap(
        colors=cores,
        index=bordas,
        vmin=bordas[0],
        vmax=bordas[-1],
    )   # usado só para calcular a cor de cada círculo; a legenda é a legenda_html()

    # Prepara as colunas que vão para o GeoJSON
    pontos = dados.copy()
    pontos["RAIO"] = raio(pontos["TOTAL"], pontos["TOTAL"].max())
    # Limita o valor ao intervalo da escala (na opção B, 9,6% vira 10%)
    pontos["PCT_COR"] = pontos["PCT_LULA"].clip(bordas[0], bordas[-1] - 1e-9)
    # Cor de cada círculo. Empate exato (50%) cairia na classe 50–60 e pareceria vitória do Lula,
    # então recebe uma cor neutra própria: branco, o centro da paleta divergente.
    # (Testamos cinza, mas ele se confundia com o mapa de fundo, que também é cinza.)
    empate = pontos["LULA"] == pontos["FLAVIO"]
    pontos["COR"] = pontos["PCT_COR"].apply(escala)
    pontos.loc[empate, "COR"] = cor_empate
    pontos["CARTAO"] = pontos.apply(cartao_html, axis=1)
    # Desenha os maiores primeiro, para os pequenos ficarem por cima e continuarem acessíveis
    pontos = pontos.sort_values("TOTAL", ascending=False)

    # Tabela -> GeoDataFrame de pontos (longitude = x, latitude = y; EPSG:4326 = lat/long comum)
    gdf = gpd.GeoDataFrame(
        pontos[["CARTAO", "RAIO", "COR"]],
        geometry=gpd.points_from_xy(pontos["NR_LONGITUDE"], pontos["NR_LATITUDE"]),
        crs="EPSG:4326",
    )

    # Mapa base: o mesmo em todos os mapas (função mapa_base, no início do notebook)
    m = mapa_base()

    folium.GeoJson(
        gdf,
        name="2º turno",
        marker=folium.CircleMarker(),   # cada ponto do GeoJSON vira um círculo
        style_function=lambda f: {
            "radius": f["properties"]["RAIO"],
            "fill": True,               # o CircleMarker vem sem preenchimento por padrão
            "fillColor": f["properties"]["COR"],
            "fillOpacity": 0.85,
            "color": "#555555",         # borda cinza: deixa visíveis os círculos de cor clara (perto de 50%)
            "weight": 0.8,
        },
        highlight_function=lambda f: {"weight": 3, "color": "#000000"},   # destaca o círculo sob o mouse
        tooltip=folium.GeoJsonTooltip(
            fields=["CARTAO"],
            labels=False,               # sem o rótulo "CARTAO:", só o conteúdo
            sticky=True,
            style=ESTILO_TOOLTIP,
        ),
    ).add_to(m)

    m.get_root().html.add_child(folium.Element(legenda_html(bordas, cores, titulo, cor_empate if empate.any() else None)))   # legenda
    # Enquadra o mapa em todos os pontos: total_bounds = [oeste, sul, leste, norte]
    oeste, sul, leste, norte = gdf.total_bounds
    m.fit_bounds([[sul, oeste], [norte, leste]])
    return m

In [22]:
# Quantos locais caem em cada classe? Ajuda a comparar as duas opções
print("Opção A:")
print(pd.cut(por_local["PCT_LULA"], BORDAS_A).value_counts().sort_index())
print()
print("Opção B:")
print(pd.cut(por_local["PCT_LULA"].clip(10, 90), BORDAS_B, include_lowest=True).value_counts().sort_index())

Opção A:
PCT_LULA
(0, 30]       2
(30, 40]      3
(40, 50]     31
(50, 60]     15
(60, 70]      9
(70, 100]     2
Name: count, dtype: int64

Opção B:
PCT_LULA
(9.999, 20.0]     1
(20.0, 30.0]      1
(30.0, 40.0]      3
(40.0, 50.0]     31
(50.0, 60.0]     15
(60.0, 70.0]      9
(70.0, 80.0]      2
(80.0, 90.0]      0
Name: count, dtype: int64


In [23]:
from pathlib import Path
Path("mapas").mkdir(exist_ok=True)

mapa_2t = mapa_2turno(por_local, BORDAS_A, RDBU_6, "2º turno – % dos votos válidos em Lula")
mapa_2t.save("mapas/mapa_2turno.html")
mapa_2t

#### Relatório – Mapa do 2º turno

**O fato**
- **Empate técnico:** Flávio 50,1% × Lula 49,9% dos votos válidos, diferença de 133 votos (brancos e nulos: 3,6%).
- **Do 1º para o 2º turno, o movimento foi só a favor de Lula:** ele subiu de 41,9% para 49,9% dos válidos, manteve os 19 prédios em que liderou e virou outros 7 (mais 2 empates); nenhum prédio passou de Lula para Flávio.
- **Flávio venceu em mais locais (34 × 26, e 2 empates), mas Lula venceu em locais menores** (mediana de 572 votos, contra 892).
- **Na cidade, o padrão é leste × oeste do centro histórico:** a leste (Matosinhos, Centro, Fábricas, Pio XII), Flávio teve 52,3%; a oeste (Tejuco, Senhor dos Montes, Bonfim, São Geraldo), Lula teve 56,6%.
- **Sede e distritos têm resultado parecido** (Lula 50,2% × 48,5%), **mas os extremos estão na zona rural:** Capela do Zueira (9,6% Lula) e Canela (73,1%).
- **UFSJ:** os campi do centro votaram mais em Lula (Dom Bosco 60,3%, Santo Antônio 53,3%); o CTAN, fora do centro, em Flávio (54,8%).
- *Ressalva:* o mapa mostra onde as pessoas votam, não onde moram.

**O design**
- **Paleta divergente RdBu** (vermelho = Lula/PT, azul = Flávio/PL), com o centro neutro fixado em 50% (empate). A escala automática colocaria o centro em 47,4%.
- **6 classes** simétricas (até 30 | 30–40 | 40–50 ‖ 50–60 | 60–70 | 70+): 86% dos votos estão entre 40% e 60%; a opção com 8 classes deixava uma classe vazia.
- **Empates exatos em branco**, para não parecerem vitória de um dos lados.
- **Círculos** (um por prédio) com **área proporcional ao total de votos**; não temos polígonos das seções.
- **Mapa base** OpenStreetMap em cinza e com opacidade 0,6: a única informação colorida é o resultado (o CartoDB Positron exige chave de API).
- **Folium:** `GeoJson` + `CircleMarker`, `GeoJsonTooltip` com cartão em HTML/CSS e legenda própria em HTML.